# SDS 07 — Graph Fundamentals, Triangles, Neighborhoods, and Clustering Coefficients

Reusable reference notebook. The pure-Python cells are for understanding and debugging. The Spark cells are intended for exam/server reuse and avoid GraphFrames or other extra Spark packages.

## Recognition cues

- immediate neighbors → degree / neighborhood
- mutually reachable groups → connected components
- two edges around a pivot → wedge / connected triple
- wedge plus closing edge → triangle
- neighbor interconnectedness → clustering coefficient
- scalable full-graph counting → edge joins / neighbor intersections, not full `collect()`

In [ ]:
from collections import defaultdict, deque
from itertools import combinations

def canonical_undirected_edges(edges):
    out=set()
    for u,v in edges:
        if u is None or v is None or u==v:
            continue
        a,b=sorted((u,v))
        out.add((a,b))
    return sorted(out)

def adjacency(edges):
    adj=defaultdict(set)
    for u,v in canonical_undirected_edges(edges):
        adj[u].add(v); adj[v].add(u)
    return adj


In [ ]:
toy_edges=[('A','B'),('A','C'),('B','C'),('B','D'),('C','D'),('D','E')]
adj=adjacency(toy_edges)
print({v: sorted(nbrs) for v,nbrs in adj.items()})
print('degrees:', {v:len(n) for v,n in adj.items()})
assert sum(len(n) for n in adj.values()) == 2*len(canonical_undirected_edges(toy_edges))


## Exact local graph utilities

In [ ]:
def connected_components_local(edges):
    adj=adjacency(edges)
    seen=set(); comps=[]
    for s in sorted(adj):
        if s in seen: continue
        q=deque([s]); seen.add(s); comp=[]
        while q:
            u=q.popleft(); comp.append(u)
            for v in adj[u]:
                if v not in seen:
                    seen.add(v); q.append(v)
        comps.append(sorted(comp))
    return comps

def common_neighbors(edges,u,v):
    adj=adjacency(edges)
    return adj[u] & adj[v]

def neighborhood_jaccard(edges,u,v):
    adj=adjacency(edges); a,b=adj[u],adj[v]
    union=a|b
    return len(a&b)/len(union) if union else 0.0

print('components:', connected_components_local(toy_edges))
print('common neighbors B,C:', common_neighbors(toy_edges,'B','C'))
print('neighborhood Jaccard B,C:', neighborhood_jaccard(toy_edges,'B','C'))


In [ ]:
def triangle_set(edges):
    adj=adjacency(edges)
    tris=set()
    for u in adj:
        for v,w in combinations(sorted(adj[u]),2):
            if w in adj[v]:
                tris.add(tuple(sorted((u,v,w))))
    return tris

def clustering_stats(edges):
    adj=adjacency(edges)
    tris=triangle_set(edges)
    tri_per=defaultdict(int)
    for a,b,c in tris:
        tri_per[a]+=1; tri_per[b]+=1; tri_per[c]+=1
    local={}
    wedges=0
    for v,nbrs in adj.items():
        d=len(nbrs); w=d*(d-1)//2; wedges += w
        local[v] = 0.0 if d<2 else tri_per[v]/w
    global_cc = 0.0 if wedges==0 else 3*len(tris)/wedges
    avg_local = sum(local.values())/len(local) if local else 0.0
    return {'triangles':len(tris),'wedges':wedges,'global_cc':global_cc,'local':local,'avg_local':avg_local}

st=clustering_stats(toy_edges)
print(st)
assert st['triangles']==2
assert st['wedges']==10
assert abs(st['global_cc']-0.6)<1e-12
assert abs(st['local']['A']-1.0)<1e-12


## Ordered two-hop convention — mirrors the Spark join logic

In [ ]:
def ordered_two_hop_counts(edges):
    adj=adjacency(edges)
    total=closed=0
    by_v={}
    for v,nbrs in adj.items():
        t=c=0
        for u in nbrs:
            for w in nbrs:
                if u==w: continue
                t += 1
                if w in adj[u]: c += 1
        by_v[v]=(t,c)
        total += t; closed += c
    return total,closed,by_v

total,closed,by_v=ordered_two_hop_counts(toy_edges)
print('ordered wedges:',total,'closed:',closed,'triangles:',closed//6,'CC:',closed/total)
assert total==20
assert closed==12
assert closed//6==2
assert abs(closed/total-0.6)<1e-12


## Optional approximation: uniform wedge sampling

If exact counting is too expensive and approximation is allowed, sample wedges uniformly. The fraction of sampled wedges that are closed estimates global transitivity. The cell below samples **uniformly from the explicit wedge population**, so it is pedagogical; a scalable sampler would avoid materializing all wedges.

In [ ]:
import random

def wedge_sample_estimate(edges, samples=1000, seed=42):
    adj=adjacency(edges)
    wedges=[]
    for v,nbrs in adj.items():
        for u,w in combinations(sorted(nbrs),2):
            wedges.append((u,v,w))
    if not wedges: return 0.0
    rng=random.Random(seed); closed=0
    for _ in range(samples):
        u,v,w=rng.choice(wedges)
        closed += int(w in adj[u])
    return closed/samples

print('wedge-sampling estimate:', wedge_sample_estimate(toy_edges,10000))


# Spark section
The following cells require PySpark on the exam/server environment. They are deliberately built from DataFrame operations only.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName('SDS07-GraphFundamentals').getOrCreate()


In [ ]:
def load_clickstream_links(path='pageviews.csv'):
    raw=(spark.read.option('sep','\t').option('header',False).csv(path)
         .toDF('src','dst','type','count'))
    return (raw.filter(F.col('type')=='link')
               .select('src','dst')
               .filter(F.col('src').isNotNull() & F.col('dst').isNotNull())
               .filter(F.col('src') != F.col('dst'))
               .distinct())

def symmetrize_edges(links):
    return (links.select('src','dst')
                 .union(links.select(F.col('dst').alias('src'),F.col('src').alias('dst')))
                 .distinct()
                 .cache())


## Degree and neighborhood summaries

In [ ]:
# links = load_clickstream_links('pageviews.csv')
# E = symmetrize_edges(links)
# degree = E.groupBy('src').agg(F.count('*').alias('degree'))
# degree.orderBy(F.desc('degree')).show(25, truncate=False)
# degree.select(F.avg('degree'),F.max('degree'),F.percentile_approx('degree',0.5)).show()


## Exam-ready global clustering coefficient — fully distributed

In [ ]:
def global_clustering_coefficient_spark(links):
    E = symmetrize_edges(links)

    two_hop = (
        E.alias('e1')
         .join(E.alias('e2'), F.col('e1.dst') == F.col('e2.src'))
         .filter(F.col('e1.src') != F.col('e2.dst'))
         .select(F.col('e1.src').alias('u'),
                 F.col('e1.dst').alias('v'),
                 F.col('e2.dst').alias('w'))
    )

    n_triplets = two_hop.count()
    n_closed = (
        two_hop.join(E.alias('e3'),
                     (F.col('u') == F.col('e3.src')) &
                     (F.col('w') == F.col('e3.dst')))
               .count()
    )
    triangles = n_closed // 6
    cc = n_closed / n_triplets if n_triplets else 0.0
    E.unpersist()
    return {'ordered_triplets':n_triplets,'ordered_closed':n_closed,
            'triangles':triangles,'global_cc':cc}

# result = global_clustering_coefficient_spark(links)
# print(result)


## Local clustering coefficients in Spark

In [ ]:
def local_clustering_coefficients_spark(links):
    E=symmetrize_edges(links)
    two_hop=(E.alias('e1')
        .join(E.alias('e2'),F.col('e1.dst')==F.col('e2.src'))
        .filter(F.col('e1.src') != F.col('e2.dst'))
        .select(F.col('e1.src').alias('u'),F.col('e1.dst').alias('v'),F.col('e2.dst').alias('w')))

    wedges=(two_hop.groupBy('v').agg(F.count('*').alias('ordered_wedges')))
    closed=(two_hop.join(E.alias('e3'),(F.col('u')==F.col('e3.src')) & (F.col('w')==F.col('e3.dst')))
                   .groupBy('v').agg(F.count('*').alias('closed_wedges')))
    out=(wedges.join(closed,'v','left').fillna(0,subset=['closed_wedges'])
         .withColumn('local_cc',F.col('closed_wedges')/F.col('ordered_wedges')))
    return out,E

# local_cc, E_cached = local_clustering_coefficients_spark(links)
# local_cc.orderBy(F.desc('local_cc')).show(25,truncate=False)
# E_cached.unpersist()


## Built-in Spark connected components via minimum-label propagation
This is a reference implementation for exam use when GraphFrames is unavailable. It is simple rather than production-optimal.

In [ ]:
def connected_components_label_propagation(links, max_iter=100, checkpoint_every=10):
    E=symmetrize_edges(links)
    nodes=(E.select(F.col('src').alias('id')).union(E.select(F.col('dst').alias('id'))).distinct().cache())
    labels=nodes.withColumn('label',F.col('id')).cache()

    for it in range(max_iter):
        msgs=(E.join(labels.select(F.col('id').alias('src'),'label'),'src')
               .groupBy('dst').agg(F.min('label').alias('nbr_label')))
        new_labels=(labels.join(msgs,labels.id==msgs.dst,'left')
                    .select(labels.id.alias('id'),F.least(labels.label,F.coalesce(msgs.nbr_label,labels.label)).alias('label'))
                    .cache())
        changed=(labels.alias('a').join(new_labels.alias('b'),'id')
                 .filter(F.col('a.label') != F.col('b.label')).count())
        labels.unpersist(); labels=new_labels
        if changed==0:
            break
    return labels,E,nodes


## Exam checklist

1. State directed/undirected and weighted/unweighted convention.
2. Remove nulls and self-loops; deduplicate if using a simple graph.
3. For global clustering, state `3T / connected triples`.
4. If using symmetric edges, explain ordered two-hop counting and `triangles = closed/6`.
5. Keep large tables distributed; collect only final scalars/top-k.
6. Do not confuse global transitivity with average local clustering.
7. Interpret the value using the meaning of the edges.